# Creation of variables necessaries for CRI

In [63]:
import pandas as pd
import numpy as np
from pyprojroot import here
import matplotlib.pyplot as plt
import seaborn as sns
import utils

figures_folder = here('figures/')
base_name_figures = '4contvar_'
processed_data = here('data/processed_data')


In [64]:
contracts = pd.read_feather(processed_data / 'mxc11to22_base.feather')
contracts.shape

(2308908, 38)

In [65]:
(contracts.isnull().sum() / contracts.shape[0]).sort_index(ascending=True)

RFC                                       0.357048
advertisement_date                        0.173678
advertisement_url                         0.000000
award_date                                0.595893
awards_per_tender                         0.664667
contract_code                             0.000000
contract_end_date                         0.000141
contract_initial_date                     0.000000
contract_price_mx                         0.000000
contract_signature_date                   0.591509
contract_year                             0.000000
contracting_type                          0.003249
country_contract_type                     0.029391
df_year                                   0.000000
file_code                                 0.000000
file_template                             0.000340
government_level                          0.000000
legal_framework                           0.029391
legal_fundament                           0.644734
legal_fundament_simplified     

# Creation of variables

- tender_period

- submission_period

- decision_period

- single_bidding

- call for tender publication

- buyer dependence

- benfors law

- conformity

- legal_dates_for_procedures

Controls

- tender_year

- contract_price quantiles

- buyer_type (government level)

- supply type

- buyerregion

one-hot-encoding for nans

### Single bidding

In [66]:
contracts['number_of_tenderers'].isnull().sum() / contracts.shape[0]

0.8792918557170749

In [67]:
pd.crosstab(contracts['number_of_tenderers'].isnull(), contracts['contract_year'])

contract_year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
number_of_tenderers,,,,,,,,,,,,
False,0,0,0,0,0,5,5131,40417,59955,47831,59959,65406
True,124033,167311,177946,195591,220647,231241,228936,153661,136721,115825,149927,128365


In [68]:
contracts['single_bidder'] = contracts['number_of_tenderers'].copy()
contracts['single_bidder'] = np.where(contracts['single_bidder'] == 1, 1, np.where(contracts['single_bidder'] > 1, 0, np.nan))

In [69]:
contracts['single_bidder'].isnull().sum() / contracts.shape[0]

0.8792918557170749

In [70]:
contracts['single_bidder'].value_counts(dropna=False, normalize=True)

NaN    0.879292
0.0    0.107756
1.0    0.012952
Name: single_bidder, dtype: float64

### Decision period

I will impute the value of award date first with contract signature date and then with contract initial date

In [71]:
print(contracts['award_date'].isnull().sum() / contracts.shape[0])
#impute the values of award date
contracts['award_date'] = np.where(contracts['award_date'].isnull(), contracts['contract_signature_date'], contracts['award_date'])
print(contracts['award_date'].isnull().sum() / contracts.shape[0])

0.5958929502604694
0.35734035310198586


In [72]:
print(contracts['award_date'].isnull().sum() / contracts.shape[0])
#impute the values of award date
contracts['award_date'] = np.where(contracts['award_date'].isnull(), contracts['contract_initial_date'], contracts['award_date'])
print(contracts['award_date'].isnull().sum() / contracts.shape[0])

0.35734035310198586
0.0


In [73]:
#decision period are days of difference between award date and submission deadline date
contracts['decision_period'] = (contracts['award_date'] - contracts['submission_deadline_date']).dt.days


In [74]:
contracts['decision_period'].describe()

count    1.656923e+06
mean     2.630602e+00
std      1.161466e+02
min     -3.862100e+04
25%      0.000000e+00
50%      1.000000e+00
75%      8.000000e+00
max      3.652400e+04
Name: decision_period, dtype: float64

In [75]:
#how many contracts have decision period less than 720
(contracts['decision_period'] < -720).sum() 

5177

In [76]:
#how many contracts have decision period less than 720
(contracts['decision_period'] < -365).sum() 

8680

In [77]:
#if decision period is greater than 720 days, then change it to 720 days
contracts['decision_period'] = np.where(contracts['decision_period'] > 365, 365, contracts['decision_period'])
#if decision period is less than -720 days, then change it to -720 days
contracts['decision_period'] = np.where(contracts['decision_period'] < -365, -365, contracts['decision_period'])

In [78]:
contracts['decision_period'].describe()

count    1.656923e+06
mean     5.536330e+00
std      4.447624e+01
min     -3.650000e+02
25%      0.000000e+00
50%      1.000000e+00
75%      8.000000e+00
max      3.650000e+02
Name: decision_period, dtype: float64

### Tender period 

Difference between award date and advertisement date

In [79]:
contracts['tender_period'] = (contracts['award_date'] - contracts['advertisement_date']).dt.days


In [80]:
contracts['tender_period'].describe()

count    1.907902e+06
mean    -1.939112e+01
std      1.355341e+02
min     -4.452400e+04
25%     -2.100000e+01
50%      3.000000e+00
75%      1.700000e+01
max      3.649700e+04
Name: tender_period, dtype: float64

In [81]:
(contracts['tender_period'] < -720).sum() 

6978

In [82]:
(contracts['tender_period'] < -365).sum() 

20565

In [83]:
(contracts['tender_period'] > 720).sum() 

1360

In [84]:
#limit tender period to 720 days
contracts['tender_period'] = np.where(contracts['tender_period'] > 720, 720, contracts['tender_period'])
contracts['tender_period'] = np.where(contracts['tender_period'] < -720, -720, contracts['tender_period'])

### Submission period

In [85]:
#submission period are days of difference between submission deadline date and advertisement date
contracts['submission_period'] = (contracts['submission_deadline_date'] - contracts['advertisement_date']).dt.days

In [86]:
contracts['submission_period'].describe()

count    1.592606e+06
mean    -1.273809e+01
std      8.259868e+01
min     -4.452400e+04
25%     -9.000000e+00
50%      4.000000e+00
75%      1.100000e+01
max      3.647100e+04
Name: submission_period, dtype: float64

In [87]:
contracts['submission_period'].quantile(0.01)

-294.0

In [88]:
(contracts['submission_period'] < -365).sum() 

8637

In [89]:
(contracts['submission_period'] > 365).sum() 

51

In [90]:
#if sumbission period is greater than 365 days, then change it to 365 days
contracts['submission_period'] = np.where(contracts['submission_period'] > 365, 365, contracts['submission_period'])
#if sumbission period is less than -365 days, then change it to -365 days
contracts['submission_period'] = np.where(contracts['submission_period'] < -365, -365, contracts['submission_period'])

In [91]:
contracts['submission_period'].isnull().sum() / contracts.shape[0]

0.3102341020083953

In [92]:
contracts['submission_period'].describe()

count    1.592606e+06
mean    -1.159512e+01
std      5.514391e+01
min     -3.650000e+02
25%     -9.000000e+00
50%      4.000000e+00
75%      1.100000e+01
max      3.650000e+02
Name: submission_period, dtype: float64

### Contract period

In [93]:
contracts['contract_period'] = (contracts['contract_end_date'] - contracts['contract_initial_date']).dt.days

In [94]:
contracts['contract_period'].describe()

count    2.308582e+06
mean     1.116227e+02
std      1.744841e+02
min     -6.300000e+01
25%      1.000000e+01
50%      4.200000e+01
75%      1.580000e+02
max      3.031500e+04
Name: contract_period, dtype: float64

In [95]:
contracts['contract_period'].quantile(0.999)

1679.0

In [96]:
#when contract period is greater than 0.999 quantile, then change it to that value
contracts['contract_period'] = np.where(contracts['contract_period'] >= contracts['contract_period'].quantile(0.999), contracts['contract_period'].quantile(0.999), contracts['contract_period'])

### Call for tender publication

In [97]:
#call for tender publication
contracts['call_for_tender_publication'] = np.where(contracts['advertisement_date'].notnull(), 1, 0)


In [98]:
contracts['call_for_tender_publication'].value_counts(dropna=False, normalize=True)

1    0.826322
0    0.173678
Name: call_for_tender_publication, dtype: float64

In [99]:
contracts['advertisement_date'].isnull().sum() / contracts.shape[0]

0.1736777732157366

### Buyer dependence

In [100]:
#get the absolute values
contracts['contract_price_mx'] = contracts['contract_price_mx'].abs()
#budget per buyer per year
contracts['spenditure_buyer_per_year'] = contracts.groupby(['purchasing_unit_id', 'contract_year'])['contract_price_mx'].transform('sum')
#proportion contract per buyer per year
contracts['proportion_contract_buyer_year'] = contracts['contract_price_mx'] / contracts['spenditure_buyer_per_year']
#proportion of share of supplier in annual budget of buyer
contracts['buyer_dependence'] = contracts.groupby(['supplier_name_clean', 'contract_year', 'purchasing_unit_id'])['proportion_contract_buyer_year'].transform('sum')

In [101]:
#remove columns we don't need
contracts = contracts.drop(columns=['proportion_contract_buyer_year', 'spenditure_buyer_per_year'])

### Benford law

In [102]:
#open json file mad_peryear.json
import json
with open(processed_data / 'mad_peryear.json') as json_file:
    mad_peryear = json.load(json_file)


In [103]:
#import data from feather
mad_df = pd.DataFrame(mad_peryear)
mad_df.head()
mad_df['mad'] = mad_df['mad'].astype(float)
mad_df['contract_year'] = mad_df['contract_year'].astype(int)
mad_df['purchasing_unit_id'] = mad_df['purchasing_unit_id'].astype(str)


In [104]:
# Define conditions and labels
conditions = [
    (mad_df['mad'] >= 0.000) & (mad_df['mad'] < 0.006), #close conformity
    (mad_df['mad'] >= 0.006) & (mad_df['mad'] < 0.012), #acceptable conformity
    (mad_df['mad'] >= 0.012) & (mad_df['mad'] < 0.016), #marginally acceptable
    (mad_df['mad'] >= 0.016) #no conformity
]

labels = ['3_high_conformity', '2_acceptable_conformity', '1_marginally_acceptable', '0_no_conformity']

# Assign categories
mad_df['bl_conformity'] = np.select(conditions, labels, default='Unknown')

In [105]:
#merge data
contracts = contracts.merge(mad_df, how='left', on=['contract_year', 'purchasing_unit_id'])

In [106]:
contracts['bl_conformity'].fillna('non_applicable', inplace=True)

# Controls

-Contract year (Aly chose tender year)

-Contract price deciles

-Government level (Buyer type for aly)

-Supply type

-State (Buyer region for Aly)

## Contract year

In [107]:
contracts['contract_year'].value_counts(dropna=False) / len(contracts['contract_year'])

2017    0.101376
2016    0.100154
2015    0.095563
2021    0.090903
2019    0.085181
2014    0.084711
2018    0.084056
2022    0.083923
2013    0.077069
2012    0.072463
2020    0.070880
2011    0.053719
Name: contract_year, dtype: float64

## Contract price deciles

In [108]:
contracts['contract_price_decile'] = pd.qcut(contracts['contract_price_mx'], q=10, labels=False)

## Government level

In [109]:
contracts['government_level'].value_counts(dropna=False, normalize=True)


federal_authority    0.904939
state_authority      0.067105
local_authority      0.027956
Name: government_level, dtype: float64

## Supply type

In [110]:
contracts['supply_type'].value_counts(dropna=False, normalize=True)

goods       0.563764
services    0.342605
works       0.090382
NaN         0.003249
Name: supply_type, dtype: float64

## State

In [111]:
import utils

In [112]:
state_dict = {
'coahuiladezaragoza' : 'coahuila',
'michoacandeocampo' : 'michoacan',
'noespecificadoporlauc' : np.nan
}

In [113]:
contracts['state'] = utils.clean_names(contracts, 'state_pf')
contracts['state'] = contracts['state'].replace(state_dict)

/mnt/sdb1/marti/detecting_corruption_mexico/scripts/dataset_creation/utils.py:85: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will*not* be treated as literal strings when regex=True.
  new_column = new_column.str.replace(r,'')


In [114]:
contracts['state'].value_counts(dropna=False, normalize=True).sort_index()

aguascalientes       0.006712
bajacalifornia       0.010644
bajacaliforniasur    0.009368
campeche             0.005701
chiapas              0.011386
chihuahua            0.013667
ciudaddemexico       0.205266
coahuila             0.014533
colima               0.005057
durango              0.008694
guanajuato           0.015406
guerrero             0.007475
hidalgo              0.010364
jalisco              0.023033
mexico               0.025093
michoacan            0.014680
morelos              0.015663
nayarit              0.007654
nuevoleon            0.013406
oaxaca               0.012917
puebla               0.015883
queretaro            0.010405
quintanaroo          0.008044
sanluispotosi        0.008663
sinaloa              0.013287
sonora               0.016963
tabasco              0.008827
tamaulipas           0.009185
tlaxcala             0.003069
veracruz             0.031413
yucatan              0.012055
zacatecas            0.006411
NaN                  0.419075
Name: stat

## Export

In [115]:
missing_rates = contracts.isnull().sum() / contracts.shape[0]   

In [116]:
#save missing_rates in excel
#missing_rates.to_excel(processed_data / 'missing_rates.xlsx')

In [117]:
contracts['supplier_name_clean'].unique().shape[0]

259534

In [118]:
contracts['purchasing_unit_id'].unique().shape[0]

5036

In [119]:
#save to feather
contracts.to_feather(processed_data / 'mxc_cricomponents.feather')
